# Deteccion de Hernia Hiatal en RX Frontal
Notebook simplificado para entrenamiento y exportacion a produccion.


In [ ]:
#%pip install mlflow

In [ ]:
# Etapa 7) Inferencia de produccion (patron microservicio)

# Entrada esperada: {'image_path': '...'}
# Salida: {'ok': bool, 'result': {...}} o {'ok': False, 'error': '...'}

from typing import Any
from pathlib import Path
import json

import torch
import torchvision.transforms as T
from torch import nn
from torchvision.models import densenet121
from PIL import Image, ImageOps


DEFAULT_MEAN = [0.485, 0.456, 0.406]
DEFAULT_STD = [0.229, 0.224, 0.225]
DEFAULT_TAM_IMAGEN = 512
DEFAULT_DROPOUT = 0.25


RAIZ = Path.cwd().resolve()
if RAIZ.name.lower() == 'jupyter':
    RAIZ = RAIZ.parent

RUTA_BUNDLE_PRODUCCION = RAIZ / 'app_hernia' / 'model' / 'production_bundle.pt'


class RecorteRetrocardiacoInferencia:
    def __init__(self, x1=0.2, x2=0.8, y1=0.15, y2=0.98):
        self.x1, self.x2, self.y1, self.y2 = x1, x2, y1, y2

    def __call__(self, imagen: Image.Image):
        ancho, alto = imagen.size
        return imagen.crop((int(self.x1 * ancho), int(self.y1 * alto), int(self.x2 * ancho), int(self.y2 * alto)))


def crear_modelo_inferencia(dropout: float = DEFAULT_DROPOUT):
    modelo = densenet121(weights=None)
    in_features = modelo.classifier.in_features
    modelo.classifier = nn.Sequential(nn.Dropout(p=float(dropout)), nn.Linear(in_features, 1))
    return modelo


def construir_modelos_desde_estados(estados_modelo: list[dict[str, torch.Tensor]], dispositivo: torch.device, dropout: float):
    modelos = []
    for estado in estados_modelo:
        modelo = crear_modelo_inferencia(dropout=dropout).to(dispositivo)
        modelo.load_state_dict(estado, strict=True)
        modelo.eval()
        modelos.append(modelo)
    return modelos


@torch.no_grad()
def predecir_tensor_ensemble(modelos: list[nn.Module], tensor: torch.Tensor) -> float:
    if len(modelos) == 0:
        raise RuntimeError('No hay modelos cargados para inferencia.')
    probs = [torch.sigmoid(modelo(tensor))[0, 0] for modelo in modelos]
    return float(torch.stack(probs).mean().item())


class ServicioInferenciaHiatal:
    def __init__(self, ruta_bundle: str | Path, dispositivo: torch.device | None = None):
        self.ruta_bundle = Path(ruta_bundle)
        self.dispositivo = dispositivo or (
            torch.device('cuda') if torch.cuda.is_available() else (
                torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
            )
        )

        self.modelos: list[nn.Module] = []
        self.umbral: float = 0.5
        self.tam_imagen: int = DEFAULT_TAM_IMAGEN
        self.usar_roi: bool = True
        self.usar_autocontraste: bool = True
        self.mean: list[float] = list(DEFAULT_MEAN)
        self.std: list[float] = list(DEFAULT_STD)
        self.dropout: float = DEFAULT_DROPOUT
        self.transformacion = None

        self._cargar_bundle()

    def _cargar_bundle(self):
        if not self.ruta_bundle.exists():
            raise RuntimeError(f'No existe bundle en: {self.ruta_bundle}')

        bundle = torch.load(self.ruta_bundle, map_location=self.dispositivo, weights_only=False)
        if not isinstance(bundle, dict):
            raise RuntimeError('Bundle invalido: estructura no es dict.')

        estados = bundle.get('model_state_dicts', [])
        if not isinstance(estados, list) or len(estados) == 0:
            raise RuntimeError('Bundle invalido: model_state_dicts vacio o ausente.')

        cfg = bundle.get('config', {}) if isinstance(bundle.get('config', {}), dict) else {}
        cfg_norm = cfg.get('normalizacion', {}) if isinstance(cfg.get('normalizacion', {}), dict) else {}

        self.umbral = float(bundle.get('global_threshold', bundle.get('threshold', 0.5)))
        self.tam_imagen = int(cfg.get('tam_imagen', DEFAULT_TAM_IMAGEN))
        self.usar_roi = bool(cfg.get('roi_retrocardiaco', cfg.get('usar_roi', True)))
        self.usar_autocontraste = bool(cfg.get('autocontraste', cfg.get('usar_autocontraste', True)))
        self.dropout = float(cfg.get('dropout', DEFAULT_DROPOUT))

        mean = cfg_norm.get('mean', DEFAULT_MEAN)
        std = cfg_norm.get('std', DEFAULT_STD)
        self.mean = list(mean) if isinstance(mean, (list, tuple)) and len(mean) == 3 else list(DEFAULT_MEAN)
        self.std = list(std) if isinstance(std, (list, tuple)) and len(std) == 3 else list(DEFAULT_STD)

        self.modelos = construir_modelos_desde_estados(estados, self.dispositivo, self.dropout)
        self.transformacion = self._construir_transformacion()

    def _construir_transformacion(self):
        ops = []
        if self.usar_roi:
            ops.append(RecorteRetrocardiacoInferencia())
        ops.extend([
            T.Resize((self.tam_imagen, self.tam_imagen)),
            T.ToTensor(),
            T.Normalize(mean=self.mean, std=self.std),
        ])
        return T.Compose(ops)

    @torch.no_grad()
    def inferir_por_ruta(self, image_path: str | Path) -> dict[str, Any]:
        ruta_imagen = Path(image_path)
        if not ruta_imagen.exists() or not ruta_imagen.is_file():
            raise RuntimeError(f'Ruta de imagen invalida: {ruta_imagen}')

        with Image.open(ruta_imagen) as imagen:
            gris = imagen.convert('L')
            if self.usar_autocontraste:
                gris = ImageOps.autocontrast(gris)
            rgb = Image.merge('RGB', (gris, gris, gris))

        tensor = self.transformacion(rgb).unsqueeze(0).to(self.dispositivo)
        prob = predecir_tensor_ensemble(self.modelos, tensor)
        pred = int(prob >= self.umbral)

        return {
            'image_path': str(ruta_imagen),
            'prob_hernia': float(prob),
            'threshold': float(self.umbral),
            'pred': int(pred),
            'pred_texto': 'Hernia' if pred == 1 else 'Normal',
            'n_modelos_ensemble': int(len(self.modelos)),
            'tam_imagen': int(self.tam_imagen),
            'dispositivo': str(self.dispositivo),
        }


def manejar_solicitud_inferencia(payload: dict[str, Any], servicio: ServicioInferenciaHiatal | None = None) -> dict[str, Any]:
    servicio_activo = servicio or SERVICIO_INFERENCIA

    if not isinstance(payload, dict):
        return {'ok': False, 'error': 'payload debe ser un dict con la llave image_path'}

    image_path = payload.get('image_path')
    if not isinstance(image_path, str) or not image_path.strip():
        return {'ok': False, 'error': 'image_path es obligatorio y debe ser string'}

    try:
        resultado = servicio_activo.inferir_por_ruta(image_path.strip())
        return {'ok': True, 'result': resultado}
    except Exception as exc:
        return {'ok': False, 'error': str(exc)}


def inferir_desde_ruta(image_path: str) -> dict[str, Any]:
    # Funcion minima para endpoint: recibe ruta y retorna respuesta serializable.
    return manejar_solicitud_inferencia({'image_path': image_path})


RUTA_BUNDLE_INFERENCIA = Path(RUTA_BUNDLE_PRODUCCION)
DISPOSITIVO_INFERENCIA = torch.device('cuda') if torch.cuda.is_available() else (
    torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
)

SERVICIO_INFERENCIA = ServicioInferenciaHiatal(RUTA_BUNDLE_INFERENCIA, dispositivo=DISPOSITIVO_INFERENCIA)
print(f'Servicio de inferencia listo. Bundle={RUTA_BUNDLE_INFERENCIA} | modelos={len(SERVICIO_INFERENCIA.modelos)}')

# Ejemplo de request (simulacion de microservicio)
payload_demo = {'image_path': str('G:\\My Drive\\Educacion\\UniAndes\\MAIA\\IV\\Proyecto_Desarrollo_de_Soluciones\\Micro-proyecto\\maia_proyecto_desarrollo_soluciones\\data\\images\\normal\\normal1.png')}
respuesta_demo = manejar_solicitud_inferencia(payload_demo)
print(json.dumps(respuesta_demo, indent=2))


payload_demo = {'image_path': str('G:\\My Drive\\Educacion\\UniAndes\\MAIA\\IV\\Proyecto_Desarrollo_de_Soluciones\\Micro-proyecto\\maia_proyecto_desarrollo_soluciones\\data\\images\\hernia\\hernia1.png')}
respuesta_demo = manejar_solicitud_inferencia(payload_demo)
print(json.dumps(respuesta_demo, indent=2))


Cuda  cuda   False
Servicio de inferencia listo. Bundle=G:\My Drive\Educacion\UniAndes\MAIA\IV\Proyecto_Desarrollo_de_Soluciones\Micro-proyecto\maia_proyecto_desarrollo_soluciones\app_hernia\model\production_bundle.pt | modelos=5
{
  "ok": true,
  "result": {
    "image_path": "G:\\My Drive\\Educacion\\UniAndes\\MAIA\\IV\\Proyecto_Desarrollo_de_Soluciones\\Micro-proyecto\\maia_proyecto_desarrollo_soluciones\\data\\images\\normal\\normal1.png",
    "prob_hernia": 0.011397795751690865,
    "threshold": 0.38104963302612305,
    "pred": 0,
    "pred_texto": "Normal",
    "n_modelos_ensemble": 5,
    "tam_imagen": 512,
    "dispositivo": "cpu"
  }
}
{
  "ok": true,
  "result": {
    "image_path": "G:\\My Drive\\Educacion\\UniAndes\\MAIA\\IV\\Proyecto_Desarrollo_de_Soluciones\\Micro-proyecto\\maia_proyecto_desarrollo_soluciones\\data\\images\\hernia\\hernia1.png",
    "prob_hernia": 0.9067384004592896,
    "threshold": 0.38104963302612305,
    "pred": 1,
    "pred_texto": "Hernia",
    "n_mo